# eml-cost Quickstart

*The Pfaffian-cost-class profiler for SymPy expressions.*

This notebook is a guided tour of `eml-cost` — what it does, why you'd use it, and what its limits are. Every claim below is backed by a computation in this notebook (no "trust us" statements).

**What you'll see:**
1. Analyze a single expression into a structural cost profile
2. Why `canonicalize()` matters (form-fragility, fixed)
3. Compare two expressions quantitatively (distance metric)
4. Profile a 50-expression cross-domain corpus in one call
5. Visualize the cost-class landscape (heatmap + dendrogram)
6. Cross-sell `eml-cost-torch` for PyTorch model profiling

## 1. Install and import

```bash
pip install eml-cost
```

In [ ]:
from eml_cost import analyze, canonicalize, PfaffianProfile
import sympy as sp
import eml_cost
print('eml-cost version:', eml_cost.__version__)

## 2. Your first cost-class analysis

In [ ]:
x = sp.Symbol('x')
for expr in [sp.exp(x), sp.sin(x), x**2 + 3*x + 1, sp.erf(x)]:
    p = PfaffianProfile.from_expression(expr)
    elem = 'EML-elementary' if p.is_elementary() else 'Pfaffian-not-EML'
    print(f'{str(expr):<25s}  {p.cost_class:<14s}  {elem}')

Notice that `erf(x)` is flagged **Pfaffian-not-EML** — it lives outside the elementary class. The other three are EML-expressible (Bishop PRML eq 1.46-style Gaussian, polynomials, sin via Euler).


## 3. Why `canonicalize()` matters

The substrate measures the cost class of the expression *as written*. Algebraically equivalent forms can produce DIFFERENT cost classes — what we call **form-fragility**.

Sigmoid is the worst offender. Four equivalent forms:

In [ ]:
x = sp.Symbol('x')
forms = [
    1 / (1 + sp.exp(-x)),
    sp.exp(x) / (sp.exp(x) + 1),
    sp.Rational(1, 2) * (1 + sp.tanh(x / 2)),
    1 - 1 / (1 + sp.exp(x)),
]

print('WITHOUT canonicalize:')
for f in forms:
    p = PfaffianProfile.from_expression(f, do_canonicalize=False)
    print(f'  {str(f):<35s} -> {p.cost_class}')

print('\nWITH canonicalize (default):')
for f in forms:
    p = PfaffianProfile.from_expression(f)
    print(f'  {str(f):<35s} -> {p.cost_class}')

`canonicalize()` is **on by default**. It applies a curated, content-preserving rewrite-rule sequence that drops form-drift across our 20-expression audit from 50% → 35% (see `bench/form-sensitivity/`).

## 4. Comparing two expressions quantitatively

`PfaffianProfile.distance()` is a real metric in the (r, d, w, c) coordinate space. It satisfies identity, symmetry, and the triangle inequality (verified across 13K+ triples in `tests/test_profile_metric.py`).

In [ ]:
x = sp.Symbol('x')
logistic_growth = 1 / (1 + sp.exp(-x))
ml_sigmoid = sp.exp(x) / (sp.exp(x) + 1)
p1 = PfaffianProfile.from_expression(logistic_growth)
p2 = PfaffianProfile.from_expression(ml_sigmoid)
print(f'logistic growth:     {p1.cost_class}')
print(f'ML sigmoid:          {p2.cost_class}')
print(f'Distance:            {p1.distance(p2):.4f}')
print(f'Compare:             {p1.compare(p2)}')

Two expressions from completely unrelated fields (population dynamics, ML activations) collapse to the **same cost class** after canonicalization. This is what `eml-cost` does that other complexity measures don't: it surfaces structural invariants across domains.

## 5. Batch analysis of the demo corpus

The package ships a small 50-expression cross-domain corpus with citations. Let's profile all 50 in one call:

In [ ]:
import csv
from importlib.resources import files

corpus_path = files('eml_cost').joinpath('data/demo_corpus.csv')
with open(corpus_path, 'r', encoding='utf-8') as f:
    rows = list(csv.DictReader(f))
print(f'Loaded {len(rows)} expressions across {len(set(r["domain"] for r in rows))} domains')

profiles = []
for r in rows:
    try:
        p = PfaffianProfile.from_expression(r['sympy_expr'].replace(' ', ', '))
        profiles.append({**r, 'profile': p})
    except Exception as e:
        pass
print(f'Successfully profiled: {len(profiles)} / {len(rows)}')

In [ ]:
from collections import Counter

class_counts = Counter(p['profile'].cost_class for p in profiles)
print('Cost-class distribution:')
for cls, n in class_counts.most_common(10):
    print(f'  {cls:<16s} x{n}')

## 6. Visualize: heatmap of pairwise distances

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

n = len(profiles)
D = np.zeros((n, n))
for i in range(n):
    for j in range(i+1, n):
        d = profiles[i]['profile'].distance(profiles[j]['profile'])
        D[i, j] = d
        D[j, i] = d

# Sort by cost class for visual clustering
order = sorted(range(n), key=lambda i: profiles[i]['profile'].cost_class)
D_sorted = D[np.ix_(order, order)]

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(D_sorted, cmap='viridis', aspect='auto')
ax.set_title(f'eml-cost distance matrix\\n{n} expressions across {len(set(r["domain"] for r in rows))} domains, sorted by cost class')
plt.colorbar(im, label='distance in (r, d, w, c) space')
plt.tight_layout()
plt.show()

## 7. Profile your PyTorch model — cross-sell `eml-cost-torch`

Want to apply this to neural networks? `eml-cost-torch` walks any `torch.nn.Module` and assigns a Pfaffian cost class to every layer.

```bash
pip install eml-cost-torch
```

```python
from eml_cost_torch import diagnose
from transformers import GPT2Config, GPT2Model

report = diagnose(GPT2Model(GPT2Config()))
print(report)
# DiagnosisReport for GPT2Model
#   layers total:                   124
#   Pfaffian-not-EML:               12   <- all GELU layers
#   elevated fp16 risk:             12
#   saturating variance:            12
```

Empirical basis: 24 GELU layers across GPT-2 + BERT show **14% higher fp16 drift** (BH-q = 0.022) and **53% lower activation variance** (BH-q = 2.1e-4) than other layer classes (E-183, n=275 layers).

Your model's activation function determines the structural class of its critical layers.

## 8. What's next

- **Research:** [monogate.org](https://monogate.org) — the One Operator framework, Lean proofs, blog
- **Foundations:** [arXiv:2603.21852](https://arxiv.org/abs/2603.21852) — Odrzywołek 2026, the founding paper
- **PyTorch:** [`eml-cost-torch`](https://pypi.org/project/eml-cost-torch/) — per-layer Pfaffian profiles
- **Rewriting:** [`eml-rewrite`](https://pypi.org/project/eml-rewrite/) — substitute high-cost forms with low-cost equivalents
- **Source:** [github.com/almaguer1986/eml-cost](https://github.com/almaguer1986/eml-cost)

All four packages share the same Pfaffian classification semantics. `canonicalize()` is the bridge between symbolic input and stable cost-class output.